# Import libraries

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np
from collections import defaultdict

# Preprocessing
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

# Models
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
import lightgbm as lgb
from catboost import CatBoostRegressor, Pool

# Model selection
from sklearn.model_selection import KFold, RandomizedSearchCV, GridSearchCV, cross_val_score 
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, make_scorer

# Read data

In [ ]:
train = pd.read_csv('crop_yield_train.csv')
test = pd.read_csv('crop_yield_test.csv')

# Feature engineering

In [ ]:
# SPLIT DATE
train['harvest_date'] = pd.to_datetime(train['harvest_date'])
test['harvest_date'] = pd.to_datetime(test['harvest_date'])

for df in [train, test]:
    df['harvest_month'] = df['harvest_date'].dt.month
    df['harvest_dayofyear'] = df['harvest_date'].dt.dayofyear
    df['harvest_week'] = df['harvest_date'].dt.isocalendar().week

# SOIL NORMALIZATION
soil_cols = ['soil_ph', 'soil_moisture', 'nitrogen_content', 'phosphorus_content', 'potassium_content']
for col in soil_cols:
    # Train normalization
    mean_region = train.groupby('region')[col].transform('mean')
    std_region = train.groupby('region')[col].transform('std')
    train[f'{col}_norm'] = (train[col] - mean_region) / (std_region + 1e-6)
    
    # Test normalization
    region_stats = train.groupby('region')[col].agg(['mean', 'std']).reset_index()
    test = test.merge(region_stats, on='region', how='left', suffixes=('', '_train'))
    test[f'{col}_norm'] = (test[col] - test['mean']) / (test['std'] + 1e-6)
    test.drop(['mean', 'std'], axis=1, inplace=True)

# NEW INDEXES
for df in [train, test]:
    df['rainfall_sunlight_index'] = df['total_rainfall'] * df['sunlight_hours']

# DROP DATE COLUMN
train = train.drop('harvest_date', axis=1)
test = test.drop('harvest_date', axis=1)

# Setup

In [ ]:
TARGET = 'yield_tpha'
ID_COL = 'id'

X_train = train.drop([TARGET, ID_COL], axis=1)
y_train = train[TARGET]
X_test = test.drop(ID_COL, axis=1)
X_test = X_test[X_train.columns]

categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_cols = X_train.select_dtypes(exclude=['object', 'category']).columns.tolist()

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),
    ('num', StandardScaler(), numeric_cols)
])

rmse_scorer = make_scorer(lambda y_true, y_pred: np.sqrt(mean_squared_error(y_true, y_pred)), greater_is_better=False)

cv = KFold(n_splits=5, shuffle=True, random_state=42)

# Run algorithms

## Algorithm 1 - Ridge regression

In [ ]:
print("[RIDGE REGRESSION]")

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', Ridge())
])

param_grid = {
    'regressor__alpha': [50, 100, 200, 500, 1000]
}

grid = GridSearchCV(pipeline, param_grid, scoring=rmse_scorer, cv=cv, n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)

ridge_rmse = -grid.best_score_
ridge_best = grid.best_estimator_
ridge_params = grid.best_params_
print(f"Best RMSE = {ridge_rmse:.4f}")
print("Best Parameters:", grid.best_params_)

## Algorithm 2 - Lasso regression

In [ ]:
print("[LASSO REGRESSION]")

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', Lasso(max_iter=10000))
])

param_grid = {
    'regressor__alpha': [0.001, 0.01, 0.1, 1, 10]
}

grid = GridSearchCV(pipeline, param_grid, scoring=rmse_scorer, cv=cv, n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)

lasso_rmse = -grid.best_score_
lasso_best = grid.best_estimator_
lasso_params = grid.best_params_
print(f"Best RMSE = {lasso_rmse:.4f}")
print("Best Parameters:", grid.best_params_)

## Algorithm 3 - ElasticNet

In [ ]:
print("[ElasticNet]")

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', ElasticNet(max_iter=10000))
])

param_grid = {
    'regressor__alpha': [0.01, 0.1, 1, 10],
    'regressor__l1_ratio': [0.2, 0.5, 0.8]
}

grid = GridSearchCV(pipeline, param_grid, scoring=rmse_scorer, cv=cv, n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)

enet_rmse = -grid.best_score_
enet_best = grid.best_estimator_
enet_params = grid.best_params_
print(f"Best RMSE = {enet_rmse:.4f}")
print("Best Parameters:", grid.best_params_)

## Algorithm 4 - Random Forest

In [ ]:
print("[RANDOM FOREST]")

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(random_state=42))
])

param_grid = {
    'regressor__n_estimators': [200, 500],
    'regressor__max_depth': [None, 10, 20],
    'regressor__min_samples_split': [2, 5]
}

grid = GridSearchCV(pipeline, param_grid, scoring=rmse_scorer, cv=cv, n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)

rf_rmse = -grid.best_score_
rf_best = grid.best_estimator_
rf_params = grid.best_params_
print(f"Best RMSE = {rf_rmse:.4f}")
print("Best Parameters:", grid.best_params_)

## Algorithm 5 - XGBoost

In [ ]:
print("[XGBoost]")

# Detect GPU for XGBoost
try:
    XGBRegressor(tree_method='gpu_hist', predictor='gpu_predictor')
    xgb_predictor = 'gpu_predictor'
    print("GPU detected")
except:
    xgb_predictor = 'cpu_predictor'
    print("GPU not detected")

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(
        objective='reg:squarederror',
        random_state=42,
        tree_method='hist',
        predictor=xgb_predictor,
        n_jobs=-1
    ))
])

param_grid = {
    'regressor__n_estimators': [500],
    'regressor__max_depth': [4, 6],
    'regressor__learning_rate': [0.05, 0.1],
    'regressor__subsample': [0.8, 1.0],
    'regressor__colsample_bytree': [0.8, 1.0],
    'regressor__reg_lambda': [1, 10]
}

grid = GridSearchCV(pipeline, param_grid, scoring=rmse_scorer, cv=cv, n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)

xgb_rmse = -grid.best_score_
xgb_best = grid.best_estimator_
xgb_params = grid.best_params_
print(f"Best RMSE = {xgb_rmse:.4f}")
print("Best Parameters:", grid.best_params_)

## Algorithm 6 - LightGBM

In [ ]:
print("[LightGBM]")

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', lgb.LGBMRegressor(random_state=42))
])

param_grid = {
    'regressor__n_estimators': [500],
    'regressor__max_depth': [4, 6],
    'regressor__learning_rate': [0.05, 0.1],
    'regressor__num_leaves': [31, 50],
    'regressor__subsample': [0.8, 1.0],
    'regressor__colsample_bytree': [0.8, 1.0],
    'regressor__reg_lambda': [1, 10]
}

grid = GridSearchCV(pipeline, param_grid, scoring=rmse_scorer, cv=cv, n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)

lgb_rmse = -grid.best_score_
lgb_best = grid.best_estimator_
lgb_params = grid.best_params_
print(f"Best RMSE = {lgb_rmse:.4f}")
print("Best Parameters:", grid.best_params_)

## Algorithm 7 - CatBoost

In [ ]:
print("[CatBoost]")

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', CatBoostRegressor(random_state=42, verbose=0))
])

param_grid = {
    'regressor__iterations': [500],
    'regressor__depth': [4, 6],
    'regressor__learning_rate': [0.05, 0.1],
    'regressor__l2_leaf_reg': [1, 5, 10]
}

grid = GridSearchCV(pipeline, param_grid, scoring=rmse_scorer, cv=cv, n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)

cat_rmse = -grid.best_score_
cat_best = grid.best_estimator_
cat_params = grid.best_params_
print(f"Best RMSE = {cat_rmse:.4f}")
print("Best Parameters:", grid.best_params_)

## 7 algorithms summary

In [ ]:
# Compare and summarize all results
results = {
    "Ridge": (ridge_rmse, ridge_best, ridge_params),
    "Lasso": (lasso_rmse, lasso_best, lasso_params),
    "ElasticNet": (enet_rmse, enet_best, enet_params),
    "RandomForest": (rf_rmse, rf_best, rf_params),
    "XGBoost": (xgb_rmse, xgb_best, xgb_params),
    "LightGBM": (lgb_rmse, lgb_best, lgb_params),
    "CatBoost": (cat_rmse, cat_best, cat_params)
}

print("[RMSE SUMMARY]")
for algo, (rmse, _, params) in results.items():
    print(f"{algo}: RMSE = {rmse:.4f}")
    print(f"   Best Params: {params}")

best_algo = min(results, key=lambda k: results[k][0])
best_rmse, best_model, best_params = results[best_algo]
print(f"\n Best algorithm: {best_algo}")
print(f"   RMSE = {best_rmse:.4f}")
print(f"   Best parameters: {best_params}")


# Optimize final algorithm choice: Catboost

## Read data

In [ ]:
train = pd.read_csv('crop_yield_train.csv')
test = pd.read_csv('crop_yield_test.csv')

## Feature engineering

In [ ]:
# SPLIT DATE
train['harvest_date'] = pd.to_datetime(train['harvest_date'])
test['harvest_date'] = pd.to_datetime(test['harvest_date'])

for df in [train, test]:
    df['harvest_month'] = df['harvest_date'].dt.month
    df['harvest_dayofyear'] = df['harvest_date'].dt.dayofyear
    df['harvest_week'] = df['harvest_date'].dt.isocalendar().week

# SOIL NORMALIZATION
soil_cols = ['soil_ph', 'soil_moisture', 'nitrogen_content', 'phosphorus_content', 'potassium_content']
for col in soil_cols:
    # Train normalization
    mean_region = train.groupby('region')[col].transform('mean')
    std_region = train.groupby('region')[col].transform('std')
    train[f'{col}_norm'] = (train[col] - mean_region) / (std_region + 1e-6)
    
    # Test normalization
    region_stats = train.groupby('region')[col].agg(['mean', 'std']).reset_index()
    test = test.merge(region_stats, on='region', how='left', suffixes=('', '_train'))
    test[f'{col}_norm'] = (test[col] - test['mean']) / (test['std'] + 1e-6)
    test.drop(['mean', 'std'], axis=1, inplace=True)

# NEW INDEXES
for df in [train, test]:
    df['rainfall_sunlight_index'] = df['total_rainfall'] * df['sunlight_hours']

# DROP DATE COLUMN
train = train.drop('harvest_date', axis=1)
test = test.drop('harvest_date', axis=1)

## New setup

In [ ]:
TARGET = 'yield_tpha'
ID_COL = 'id'

# Train and test sets
X_train = train.drop([TARGET, ID_COL], axis=1)
y_train = train[TARGET]
X_test = test.drop(ID_COL, axis=1)
X_test = X_test[X_train.columns]

# Identify categorical columns
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

# Convert column names → indices
categorical_features = [i for i, col in enumerate(X_train.columns) if col in categorical_cols]

# RMSE scorer
rmse_scorer = make_scorer(
    lambda y_true, y_pred: np.sqrt(mean_squared_error(y_true, y_pred)),
    greater_is_better=False
)

## Improved Catboost

In [ ]:
# CatBoost base model
cat = CatBoostRegressor(
    random_state=42,
    verbose=0,
    early_stopping_rounds=50,   # stops if no improvement for 50 rounds
    task_type='CPU',            # or 'GPU' if available
)

# Parameter space for RandomizedSearch
param_dist = {
    'iterations': [200, 300, 400],
    'depth': [4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1],
    'l2_leaf_reg': [1, 3, 5, 7],
    'border_count': [32, 64, 128]  # optional: for numeric binning
}

# Cross-validation setup
cv = KFold(n_splits=3, shuffle=True, random_state=42)  # use 3 folds for speed

# RandomizedSearchCV
grid = RandomizedSearchCV(
    estimator=cat,
    param_distributions=param_dist,
    n_iter=20,           # only 20 random combinations
    scoring=rmse_scorer,
    cv=cv,
    verbose=2,
    n_jobs=-1,
    random_state=42
)

# Fit the search
grid.fit(X_train, y_train, cat_features=categorical_features)

# Best model and RMSE
best_cat = grid.best_estimator_
train_rmse = np.sqrt(mean_squared_error(y_train, best_cat.predict(X_train)))

In [ ]:
print(f"Best CV RMSE: {-grid.best_score_:.4f}")
print(f"Train RMSE on full training set: {train_rmse:.4f}")
print("Best parameters:", grid.best_params_)

In [ ]:
# Cross validation results
for i in range(len(grid.cv_results_['params'])):
    print(f"\nModel {i+1}")

    # Parameters
    print("Params:", grid.cv_results_['params'][i])

    # Mean RMSE
    mean_rmse = -grid.cv_results_['mean_test_score'][i]
    print(f"Mean CV RMSE: {mean_rmse:.6f}")

    # Standard deviation of CV
    std_rmse = grid.cv_results_['std_test_score'][i]
    print(f"Std RMSE: {std_rmse:.6f}")

    # Individual fold RMSEs
    splits = [col for col in grid.cv_results_ if col.startswith("split") and col.endswith("_test_score")]
    for s in splits:
        fold_rmse = -grid.cv_results_[s][i]
        print(f"{s}: {fold_rmse:.6f}")
